# 01, Landing Page A/B Test Analysis

**Goal:** Determine whether a new landing page design (treatment) converts significantly better than the original (control) using two-proportion z-test and chi-square test.

**Dataset:** Synthetic landing page experiment, 10,000 visitors split 50/50 between control and treatment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import warnings, os

warnings.filterwarnings("ignore")
os.makedirs("../visuals", exist_ok=True)

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
print("Setup complete.")

Setup complete.


## 1. Generate / Load Dataset

In [2]:

N = 10_000
control_n = treatment_n = N // 2

np.random.seed(42)
control_converted = np.random.binomial(control_n,   0.10)
treatment_converted = np.random.binomial(treatment_n, 0.12)

data = {
    "group"     : ["control"] * control_n + ["treatment"] * treatment_n,
    "converted" : (
        [1] * control_converted   + [0] * (control_n   - control_converted) +
        [1] * treatment_converted + [0] * (treatment_n - treatment_converted)
    )
}
df = pd.DataFrame(data).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total visitors      : {len(df):,}")
print(f"Control   visitors  : {control_n:,}  | Converted: {control_converted:,}")
print(f"Treatment visitors  : {treatment_n:,}  | Converted: {treatment_converted:,}")
print(f"Control   CVR       : {control_converted/control_n:.2%}")
print(f"Treatment CVR       : {treatment_converted/treatment_n:.2%}")
df.head()

Total visitors      : 10,000
Control   visitors  : 5,000  | Converted: 482
Treatment visitors  : 5,000  | Converted: 603
Control   CVR       : 9.64%
Treatment CVR       : 12.06%


,group,converted
0,treatment,0
1,control,0
2,control,0
3,control,0
4,control,0


## 2. Descriptive Summary

In [3]:
summary = (
    df.groupby("group")["converted"]
    .agg(visitors="count", conversions="sum")
    .assign(cvr=lambda x: x["conversions"] / x["visitors"] * 100)
    .round({"cvr": 2})
)
print(summary)

relative_lift = (summary.loc["treatment", "cvr"] - summary.loc["control", "cvr"]) / summary.loc["control", "cvr"] * 100
print(f"\nAbsolute lift : {summary.loc['treatment', 'cvr'] - summary.loc['control', 'cvr']:.2f} pp")
print(f"Relative lift : {relative_lift:.2f}%")

           visitors  conversions    cvr
group                                  
control        5000          482   9.64
treatment      5000          603  12.06

Absolute lift : 2.42 pp
Relative lift : 25.10%


## 3. Two-Proportion Z-Test

In [4]:
alpha = 0.05

count = np.array([treatment_converted, control_converted])
nobs = np.array([treatment_n, control_n])

z_stat, p_value = proportions_ztest(count, nobs, alternative="larger")
ci_low, ci_high = proportion_confint(count[0], nobs[0], alpha=alpha, method="normal")

print("=" * 50)
print("TWO-PROPORTION Z-TEST")
print("=" * 50)
print(f"  H0 : CVR(treatment) <= CVR(control)")
print(f"  H1 : CVR(treatment) >  CVR(control)")
print(f"  Alpha           : {alpha}")
print(f"  Z-statistic     : {z_stat:.4f}")
print(f"  P-value         : {p_value:.4f}")
print(f"  Significant     : {'YES ✓' if p_value < alpha else 'NO ✗'}")
print(f"  95% CI (treatment CVR): [{ci_low:.4f}, {ci_high:.4f}]")

TWO-PROPORTION Z-TEST
  H0 : CVR(treatment) <= CVR(control)
  H1 : CVR(treatment) >  CVR(control)
  Alpha           : 0.05
  Z-statistic     : 3.8905
  P-value         : 0.0001
  Significant     : YES ✓
  95% CI (treatment CVR): [0.1116, 0.1296]


## 4. Chi-Square Test

In [5]:
contingency = np.array([
    [treatment_converted,   treatment_n - treatment_converted],
    [control_converted,     control_n   - control_converted]
])

chi2, p_chi2, dof, expected = chi2_contingency(contingency)

print("=" * 50)
print("CHI-SQUARE TEST OF INDEPENDENCE")
print("=" * 50)
print(f"  Chi2 statistic  : {chi2:.4f}")
print(f"  P-value         : {p_chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  Significant     : {'YES ✓' if p_chi2 < alpha else 'NO ✗'}")

CHI-SQUARE TEST OF INDEPENDENCE
  Chi2 statistic  : 14.8871
  P-value         : 0.0001
  Degrees of freedom: 1
  Significant     : YES ✓


## 5. Minimum Detectable Effect (MDE)

In [6]:
def calculate_mde(baseline_rate, n_per_group, alpha=0.05, power=0.80):
    """Calculate the smallest effect this experiment can reliably detect."""
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    p = baseline_rate
    se = np.sqrt(2 * p * (1 - p) / n_per_group)
    mde = (z_alpha + z_beta) * se
    return mde

baseline_cvr = control_converted / control_n
mde = calculate_mde(baseline_cvr, control_n)

print("=" * 50)
print("MINIMUM DETECTABLE EFFECT")
print("=" * 50)
print(f"  Baseline CVR    : {baseline_cvr:.4f} ({baseline_cvr:.2%})")
print(f"  N per group     : {control_n:,}")
print(f"  Alpha           : 0.05  |  Power: 80%")
print(f"  MDE (absolute)  : {mde:.4f} ({mde:.2%})")
print(f"  MDE (relative)  : {mde/baseline_cvr:.2%}")

MINIMUM DETECTABLE EFFECT
  Baseline CVR    : 0.0964 (9.64%)
  N per group     : 5,000
  Alpha           : 0.05  |  Power: 80%
  MDE (absolute)  : 0.0165 (1.65%)
  MDE (relative)  : 17.15%


## 6. Sample Size Planning

In [7]:
def calculate_sample_size(baseline_rate, mde, alpha=0.05, power=0.80):
    """Calculate required sample size per group for given MDE."""
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    p1 = baseline_rate
    p2 = baseline_rate + mde
    p_bar = (p1 + p2) / 2
    n = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / (p2 - p1)**2
    return int(np.ceil(n))

mde_values = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]
sample_sizes = [calculate_sample_size(baseline_cvr, m) for m in mde_values]

ss_df = pd.DataFrame({"MDE (abs)": mde_values, "N per group": sample_sizes})
ss_df["MDE (rel)"] = ss_df["MDE (abs)"] / baseline_cvr
ss_df["Total N"]   = ss_df["N per group"] * 2
print(ss_df.to_string(index=False))

 MDE (abs)  N per group  MDE (rel)  Total N
     0.005        55958   0.051867   111916
     0.010        14303   0.103734    28606
     0.015         6495   0.155602    12990
     0.020         3731   0.207469     7462
     0.025         2437   0.259336     4874
     0.030         1726   0.311203     3452


## 7. Visualizations

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

groups = ["Control", "Treatment"]
cvrs = [control_converted/control_n, treatment_converted/treatment_n]
colors = ["#4C72B0", "#DD8452"]
bars = axes[0, 0].bar(groups, [c*100 for c in cvrs], color=colors, edgecolor="white", linewidth=1.2, width=0.5)
for bar, val in zip(bars, cvrs):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f"{val:.2%}", ha="center", fontsize=11, fontweight="bold")
axes[0, 0].set_title("Conversion Rate: Control vs Treatment", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Conversion Rate (%)")
axes[0, 0].set_ylim(0, max(cvrs)*100 * 1.3)
axes[0, 0].axhline(baseline_cvr*100, linestyle="--", color="gray", linewidth=1, label="Baseline")
axes[0, 0].legend()

x = np.linspace(-4, 4, 500)
axes[0, 1].plot(x, norm.pdf(x), color="#333", linewidth=2)
x_reject = np.linspace(norm.ppf(1 - alpha), 4, 200)
axes[0, 1].fill_between(x_reject, norm.pdf(x_reject), alpha=0.3, color="red", label=f"Rejection region (α={alpha})")
axes[0, 1].axvline(z_stat, color="#DD8452", linewidth=2, linestyle="--", label=f"Z-stat = {z_stat:.3f}")
axes[0, 1].set_title("Z-Test: Standard Normal Distribution", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Z-Score")
axes[0, 1].legend(fontsize=9)

axes[1, 0].plot([m*100 for m in mde_values], sample_sizes, "o-", color="#4C72B0", linewidth=2, markersize=7)
axes[1, 0].axhline(control_n, linestyle="--", color="gray", label=f"Current N = {control_n:,}")
axes[1, 0].set_xlabel("Minimum Detectable Effect (%)")
axes[1, 0].set_ylabel("Required N per Group")
axes[1, 0].set_title("Sample Size vs MDE", fontsize=12, fontweight="bold")
axes[1, 0].legend()

ctrl_ci = proportion_confint(control_converted,   control_n,   alpha=alpha, method="normal")
trt_ci = proportion_confint(treatment_converted, treatment_n, alpha=alpha, method="normal")
y_pos = [0, 1]
means = [control_converted/control_n, treatment_converted/treatment_n]
err_low = [means[0]-ctrl_ci[0], means[1]-trt_ci[0]]
err_high = [ctrl_ci[1]-means[0], trt_ci[1]-means[1]]
axes[1, 1].errorbar(means, y_pos, xerr=[err_low, err_high],
                    fmt="o", markersize=10, color="#4C72B0",
                    ecolor="#4C72B0", elinewidth=2, capsize=8)
axes[1, 1].set_yticks(y_pos)
axes[1, 1].set_yticklabels(["Control", "Treatment"])
axes[1, 1].set_xlabel("Conversion Rate")
axes[1, 1].set_title("95% Confidence Intervals", fontsize=12, fontweight="bold")
axes[1, 1].axvline(means[0], linestyle="--", color="gray", linewidth=1)

plt.suptitle("Landing Page A/B Test, Full Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/ab_test_analysis.png", bbox_inches="tight")
plt.show()
print("Saved: ../visuals/ab_test_analysis.png")

Saved: ../visuals/ab_test_analysis.png


## 8. Conclusion

In [9]:
print("=" * 55)
print("EXPERIMENT CONCLUSION")
print("=" * 55)
print(f"  Control CVR     : {control_converted/control_n:.2%}")
print(f"  Treatment CVR   : {treatment_converted/treatment_n:.2%}")
print(f"  Absolute lift   : {(treatment_converted/treatment_n - control_converted/control_n):.2%}")
print(f"  Relative lift   : {relative_lift:.2f}%")
print(f"  Z-statistic     : {z_stat:.4f}")
print(f"  P-value (z)     : {p_value:.4f}")
print(f"  P-value (chi2)  : {p_chi2:.4f}")
print(f"  Statistically significant at α=0.05: {'YES ✓' if p_value < alpha else 'NO ✗'}")
print()
if p_value < alpha:
    print("  → Reject H0. The new landing page performs significantly better.")
    print("    Recommend rolling out the treatment page.")
else:
    print("  → Fail to reject H0. No statistically significant improvement detected.")

EXPERIMENT CONCLUSION
  Control CVR     : 9.64%
  Treatment CVR   : 12.06%
  Absolute lift   : 2.42%
  Relative lift   : 25.10%
  Z-statistic     : 3.8905
  P-value (z)     : 0.0001
  P-value (chi2)  : 0.0001
  Statistically significant at α=0.05: YES ✓

  → Reject H0. The new landing page performs significantly better.
    Recommend rolling out the treatment page.
